# Research Paper Summarization — PEGASUS (Abstractive Baseline)



> **Dataset:** `ccdv/pubmed-summarization` (PubMed) dari Hugging Face  
> **Model:** `google/pegasus-pubmed` (pre-trained langsung di PubMed!)

---
## 0. Install Dependencies

In [ ]:
!pip install transformers datasets rouge-score sentencepiece accelerate -q
# google/pegasus-pubmed butuh sentencepiece untuk tokenizer-nya

---
## 1. Dataset Preparation

Sesuai proposal: pakai subset **PubMed** dari Scientific Papers Dataset.  
Data dipetakan menjadi `input_text` dan `target_text`.  
Split mengikuti bawaan dataset (train/validation/test).

In [ ]:
from datasets import load_dataset
import pandas as pd

print("Loading dataset PubMed dari Hugging Face...")
print("Ref: https://huggingface.co/datasets/ccdv/pubmed-summarization")

dataset = load_dataset("ccdv/pubmed-summarization")

print("\nDataset splits tersedia:")
print(dataset)
print(f"\nJumlah data:")
print(f"  Train      : {len(dataset['train']):,}")
print(f"  Validation : {len(dataset['validation']):,}")
print(f"  Test       : {len(dataset['test']):,}")


In [ ]:
# Lihat contoh satu data
sample = dataset['train'][0]
print("=== Contoh Data (Train[0]) ===")
print(f"\nKolom tersedia: {list(sample.keys())}")
print(f"\nPanjang article : {len(sample['article'].split())} kata")
print(f"Panjang abstract: {len(sample['abstract'].split())} kata")
print(f"\n--- Article (200 karakter pertama) ---")
print(sample['article'][:200])
print(f"\n--- Abstract ---")
print(sample['abstract'][:300])

In [ ]:
# Mapping ke input_text dan target_text sesuai proposal
def map_columns(example):
    return {
        "input_text": example["article"],
        "target_text": example["abstract"]
    }

dataset = dataset.map(map_columns, remove_columns=["article", "abstract"])

print("Kolom setelah mapping:", dataset['train'].column_names)
print("Contoh input_text (50 kata pertama):")
print(" ".join(dataset['train'][0]['input_text'].split()[:50]))


In [ ]:
DEBUG_MODE = True
DEBUG_SIZE = 500
EVAL_SIZE  = 500

if DEBUG_MODE:
    train_data = dataset['train'].select(range(DEBUG_SIZE))
    val_data   = dataset['validation'].select(range(EVAL_SIZE))
    test_data  = dataset['test'].select(range(EVAL_SIZE))
    print(f"[DEBUG MODE] Subset kecil:")
else:
    train_data = dataset['train']
    val_data   = dataset['validation']
    test_data  = dataset['test']
    print(f"[FULL MODE] Seluruh dataset:")

print(f"  Train : {len(train_data):,}")
print(f"  Val   : {len(val_data):,}")
print(f"  Test  : {len(test_data):,}")


In [ ]:
# Simpan dataset lokal supaya tidak perlu download ulang
SAVE_PATH = "./scientific_papers_pubmed"

import os
if not os.path.exists(SAVE_PATH):
    dataset.save_to_disk(SAVE_PATH)
    print(f"Dataset disimpan ke: {SAVE_PATH}")
else:
    print(f"Dataset sudah ada di: {SAVE_PATH} — skip saving")


---
## 2. Text Pre-processing

Sesuai proposal:
- **Cleaning**: gabung kalimat, hapus simbol tidak penting
- **Normalization**: rapikan spasi dan format teks
- **Tokenization**: pakai tokenizer model, atur `max_input_length` dan `max_target_length`

> ⚠️ Tidak menggunakan stemming/lemmatization/stopword removal karena Transformer butuh struktur bahasa utuh.

In [ ]:
import re

def clean_text(text):
    """
    Cleaning & Normalization sesuai proposal:
    - Gabung kalimat jadi satu teks utuh
    - Hapus simbol tidak penting
    - Rapikan spasi/tab/newline
    """
    # Handle list of sentences (kalau input berupa list)
    if isinstance(text, list):
        text = " ".join(text)

    # Hapus karakter khusus yang tidak bermakna (bukan alfanumerik/punctuation ilmiah)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)

    # Normalisasi spasi, tab, newline berlebih
    text = re.sub(r'\s+', ' ', text)

    # Trim
    text = text.strip()

    return text

# Test cleaning
sample_raw = dataset['train'][0]['input_text']
sample_clean = clean_text(sample_raw)
print("Sebelum cleaning (50 karakter pertama):")
print(repr(sample_raw[:100]))
print("\nSetelah cleaning (50 karakter pertama):")
print(repr(sample_clean[:100]))

In [ ]:
from transformers import PegasusTokenizer

# ✅ Ganti ke versi base yang jauh lebih kecil
MODEL_NAME = "google/pegasus-xsum"
tokenizer = PegasusTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LENGTH  = 512   # turunin juga dari 1024 → 512 biar hemat VRAM
MAX_TARGET_LENGTH = 128   # dari 256 → 128

print(f"Tokenizer: {MODEL_NAME}")
print(f"Max input length : {MAX_INPUT_LENGTH}")
print(f"Max target length: {MAX_TARGET_LENGTH}")


In [ ]:
def preprocess(examples):
    """
    Full preprocessing pipeline sesuai proposal:
    Cleaning → Normalization → Tokenization
    """
    # Cleaning & Normalization
    inputs  = [clean_text(txt) for txt in examples["input_text"]]
    targets = [clean_text(txt) for txt in examples["target_text"]]

    # Tokenization input
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True
    )

    # Tokenization target
    labels = tokenizer(
       text_target=targets,
       max_length=MAX_TARGET_LENGTH,
       padding="max_length",
       truncation=True
    )

    # Set padding token jadi -100 agar tidak dihitung dalam loss
    labels_ids = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels_ids
    return model_inputs

# Apply preprocessing
print("Preprocessing train data...")
tokenized_train = train_data.map(preprocess, batched=True, remove_columns=train_data.column_names)
print("Preprocessing validation data...")
tokenized_val = val_data.map(preprocess, batched=True, remove_columns=val_data.column_names)
print("Preprocessing test data...")
tokenized_test = test_data.map(preprocess, batched=True, remove_columns=test_data.column_names)

print(f"\nSelesai! Kolom tokenized: {tokenized_train.column_names}")

---
## 3. Model Training
 

Parameter training: learning rate, batch size, epoch dikonfigurasi di sini.

---
## 3. Model Training

In [ ]:
from transformers import PegasusForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import torch, os

os.environ["WANDB_DISABLED"] = "true"

print(f"Loading model: {MODEL_NAME}")
model = PegasusForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.use_cache = False

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nJumlah parameter: {model.num_parameters():,}")


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-pubmed-results",

    num_train_epochs=3,
    learning_rate=5e-5,

    # ✅ Turunin batch size
    per_device_train_batch_size=1,   # dari 2 → 1
    per_device_eval_batch_size=1,    # dari 2 → 1

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # logging_dir="./logs",
    logging_steps=10,
    save_total_limit=2,

    # ✅ Naikin gradient accumulation untuk kompensasi batch kecil
    gradient_accumulation_steps=8,   # dari 4 → 8 (efektifnya tetap batch 8)
    
    fp16=torch.cuda.is_available(),
    warmup_steps=100,
    weight_decay=0.01,
    report_to='none',  # disable wandb
)

In [ ]:
# Data collator untuk seq2seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

# Setup Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,  # pakai processing_class (tokenizer deprecated)
    data_collator=data_collator,
)

print("Trainer siap!")
print(f"Total training steps: {len(tokenized_train) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")


In [ ]:
# ============================================================
# TRAINING
# Sesuai proposal: dilatih menggunakan data train,
# dipantau menggunakan validation set
# ============================================================
print("Mulai training BART...")
print("="*50)

train_result = trainer.train()

print("\nTraining selesai!")
print(f"Training loss akhir: {train_result.training_loss:.4f}")
print(f"Total runtime      : {train_result.metrics['train_runtime']:.1f} detik")

In [ ]:
MODEL_SAVE_PATH = "./pegasus-pubmed-finetuned"
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"Model disimpan ke: {MODEL_SAVE_PATH}")


---
## 4. Inference — Generate Ringkasan

Sesuai proposal: setelah training, dilakukan **inferensi** untuk menghasilkan ringkasan dari data test.

In [ ]:
from transformers import PegasusForConditionalGeneration, PegasusTokenizer

# Load model fine-tuned langsung (tanpa pipeline)
model_inf = PegasusForConditionalGeneration.from_pretrained("./pegasus-pubmed-finetuned")
tokenizer_inf = PegasusTokenizer.from_pretrained("./pegasus-pubmed-finetuned")
model_inf = model_inf.to(device)
model_inf.eval()

print("Model inferensi siap!")


In [ ]:
import torch

def generate_summary(text, max_length=256, min_length=50):
    inputs = tokenizer_inf(
        text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        summary_ids = model_inf.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_length,
            min_length=min_length,
            no_repeat_ngram_size=3,
            num_beams=2,
            early_stopping=True
        )

    return tokenizer_inf.decode(summary_ids[0], skip_special_tokens=True)

# Test inferensi pada satu sampel
sample = test_data[0]
print("--- Generated Summary ---")
print(generate_summary(sample["input_text"]))
print("\n--- Reference Abstract ---")
print(sample["target_text"])


In [ ]:
# Generate ringkasan untuk seluruh test set
print("Generating ringkasan untuk semua data test...")

generated_summaries = []
reference_summaries = []

for i, sample in enumerate(test_data):
    gen = generate_summary(sample['input_text'])
    generated_summaries.append(gen)
    reference_summaries.append(sample['target_text'])

    if (i + 1) % 20 == 0:
        print(f"  Progress: {i+1}/{len(test_data)}")

print(f"\nSelesai! Total ringkasan: {len(generated_summaries)}")

---
## 5. Evaluation

Sesuai proposal: evaluasi menggunakan **ROUGE-1, ROUGE-2, ROUGE-L** sebagai metrik utama,
dilengkapi dengan **analisis kualitatif** beberapa sampel.

In [ ]:
from rouge_score import rouge_scorer
import numpy as np

def compute_rouge(predictions, references):
    """
    Hitung ROUGE-1, ROUGE-2, ROUGE-L sesuai proposal.
    Return rata-rata F1 score.
    """
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )

    scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        scores["rouge1"].append(score["rouge1"].fmeasure)
        scores["rouge2"].append(score["rouge2"].fmeasure)
        scores["rougeL"].append(score["rougeL"].fmeasure)

    return {
        "ROUGE-1": round(np.mean(scores["rouge1"]) * 100, 2),
        "ROUGE-2": round(np.mean(scores["rouge2"]) * 100, 2),
        "ROUGE-L": round(np.mean(scores["rougeL"]) * 100, 2),
    }

# Hitung ROUGE
print("Menghitung ROUGE scores...")
rouge_scores = compute_rouge(generated_summaries, reference_summaries)

print("\n" + "="*40)
print("       HASIL EVALUASI — BART")
print("="*40)
for metric, score in rouge_scores.items():
    print(f"  {metric} : {score:.2f}")
print("="*40)
print("(Skala 0-100, semakin tinggi semakin baik)")

In [ ]:
# Simpan hasil ke DataFrame untuk comparative analysis nanti
results_df = pd.DataFrame([{
    "Model": "PEGASUS (google/pegasus-pubmed)",
    "ROUGE-1": rouge_scores["ROUGE-1"],
    "ROUGE-2": rouge_scores["ROUGE-2"],
    "ROUGE-L": rouge_scores["ROUGE-L"],
    "Tipe": "Abstractive Baseline",
    "Dataset": "PubMed",
    "Test Samples": len(test_data)
}])

results_df.to_csv("pegasus_rouge_results.csv", index=False)
print("Hasil disimpan ke: pegasus_rouge_results.csv")
print("\n(File ini akan digunakan untuk Comparative Analysis di tahap akhir)")
results_df

---
## 📊 Visualisasi & Metric Evaluation

Visualisasi hasil evaluasi model: ROUGE scores, distribusi per sampel, analisis panjang, dan training curves.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import numpy as np
from rouge_score import rouge_scorer as rs_mod

# ── Hitung score per sampel ──
sc = rs_mod.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
per_sample = {'ROUGE-1': [], 'ROUGE-2': [], 'ROUGE-L': []}
for pred, ref in zip(generated_summaries, reference_summaries):
    s = sc.score(ref, pred)
    per_sample['ROUGE-1'].append(s['rouge1'].fmeasure * 100)
    per_sample['ROUGE-2'].append(s['rouge2'].fmeasure * 100)
    per_sample['ROUGE-L'].append(s['rougeL'].fmeasure * 100)

# ── Plot 1: ROUGE Bar Chart ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'ROUGE Evaluation — {MODEL_NAME}', fontsize=13, fontweight='bold')

metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
scores  = [rouge_scores[m] for m in metrics]
colors  = ['#4C72B0', '#DD8452', '#55A868']

bars = axes[0].bar(metrics, scores, color=colors, width=0.5, edgecolor='white')
axes[0].set_ylim(0, max(scores) * 1.35)
axes[0].set_ylabel('F1 Score (%)')
axes[0].set_title('ROUGE Scores (Average)')
axes[0].grid(True, axis='y', alpha=0.3)
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{score:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# ── Plot 2: Distribusi ROUGE-1 (Histogram) ──
axes[1].hist(per_sample['ROUGE-1'], bins=20, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[1].axvline(np.mean(per_sample['ROUGE-1']), color='tomato', linestyle='--',
               linewidth=2, label=f'Mean: {np.mean(per_sample["ROUGE-1"]):.2f}')
axes[1].set_xlabel('ROUGE-1 F1 Score (%)')
axes[1].set_ylabel('Jumlah Sampel')
axes[1].set_title('Distribusi ROUGE-1 per Sampel')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./viz_rouge_bar_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz_rouge_bar_hist.png")


In [ ]:
# ── Plot 3: Boxplot semua metrik ──
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(f'Score Distribution — {MODEL_NAME}', fontsize=12, fontweight='bold')

bp = ax.boxplot(list(per_sample.values()), labels=list(per_sample.keys()),
                patch_artist=True, medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#4C72B0', '#DD8452', '#55A868']):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('F1 Score (%)')
ax.grid(True, axis='y', alpha=0.3)
for i, (m, vals) in enumerate(per_sample.items(), 1):
    med = np.median(vals)
    ax.text(i, med + 0.3, f'{med:.2f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('./viz_rouge_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz_rouge_boxplot.png")


In [ ]:
# ── Plot 4: Panjang Input vs ROUGE-1 & Generated vs Reference ──
input_lengths = [len(sample['input_text'].split()) for sample in test_data]
gen_lengths   = [len(s.split()) for s in generated_summaries]
ref_lengths   = [len(s.split()) for s in reference_summaries]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Length Analysis — {MODEL_NAME}', fontsize=13, fontweight='bold')

# Scatter: input length vs ROUGE-1
axes[0].scatter(input_lengths, per_sample['ROUGE-1'], alpha=0.4, color='#4C72B0', s=18)
z = np.polyfit(input_lengths, per_sample['ROUGE-1'], 1)
x_line = np.linspace(min(input_lengths), max(input_lengths), 100)
axes[0].plot(x_line, np.poly1d(z)(x_line), color='tomato', linewidth=2,
            linestyle='--', label='Trend')
axes[0].set_xlabel('Panjang Input (kata)')
axes[0].set_ylabel('ROUGE-1 (%)')
axes[0].set_title('Panjang Input vs ROUGE-1')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bar: generated vs reference length
avg_gen, avg_ref = np.mean(gen_lengths), np.mean(ref_lengths)
bars = axes[1].bar(['Generated\nSummary', 'Reference\nAbstract'],
                   [avg_gen, avg_ref],
                   color=['#4C72B0', '#55A868'], width=0.4, edgecolor='white')
axes[1].set_ylabel('Rata-rata Panjang (kata)')
axes[1].set_title('Panjang Generated vs Reference')
axes[1].grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, [avg_gen, avg_ref]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('./viz_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz_length_analysis.png")


In [ ]:
# ── Plot 5: Training & Validation Loss Curve ──
history = trainer.state.log_history
train_steps, train_losses, eval_epochs, eval_losses = [], [], [], []

for entry in history:
    if 'loss' in entry and 'eval_loss' not in entry:
        train_steps.append(entry['step'])
        train_losses.append(entry['loss'])
    if 'eval_loss' in entry:
        eval_epochs.append(entry['epoch'])
        eval_losses.append(entry['eval_loss'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Training Curves — {MODEL_NAME}', fontsize=13, fontweight='bold')

axes[0].plot(train_steps, train_losses, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss per Step')
axes[0].grid(True, alpha=0.3)

if eval_losses:
    axes[1].plot(eval_epochs, eval_losses, color='tomato', linewidth=2,
                marker='o', markersize=7, label='Val Loss')
    for ep, loss in zip(eval_epochs, eval_losses):
        axes[1].annotate(f'{loss:.4f}', (ep, loss),
                        textcoords='offset points', xytext=(0, 8),
                        ha='center', fontsize=9)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Validation Loss per Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No eval loss logged', ha='center', va='center',
                transform=axes[1].transAxes, fontsize=12, color='gray')

plt.tight_layout()
plt.savefig('./viz_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: viz_training_curves.png")


---
## 📺 Tampilkan Visualisasi

In [ ]:
from IPython.display import display, Image
import glob

# ── Tampilkan semua visualisasi ──
viz_files = sorted(glob.glob('./viz_*.png'))
print(f"Total visualisasi: {len(viz_files)}")
print("=" * 50)

for f in viz_files:
    name = f.replace('./viz_', '').replace('.png', '').replace('_', ' ').title()
    print(f"\n📊 {name}")
    display(Image(filename=f, width=900))


---
## 💾 Simpan Hasil ke Google Drive

---
## 6. Evaluasi Kualitatif

Sesuai proposal: cek apakah ringkasan mencakup informasi penting:  
**tujuan penelitian, metode, hasil utama, dan kesimpulan**.  
Deteksi **hallucination** dan kehilangan konteks.

In [ ]:
# Analisis kualitatif pada 3 sampel
NUM_QUALITATIVE = 3

print("=" * 60)
print("         EVALUASI KUALITATIF — PEGASUS")
print("=" * 60)

for idx in range(NUM_QUALITATIVE):
    print(f"\n📄 SAMPEL #{idx+1}")
    print("-" * 60)

    print("[INPUT — 100 kata pertama]:")
    print(" ".join(test_data[idx]['input_text'].split()[:100]) + "...")

    print("\n[GENERATED SUMMARY]:")
    print(generated_summaries[idx])

    print("\n[REFERENCE ABSTRACT]:")
    print(reference_summaries[idx][:400] + "...")

    # Hitung ROUGE per sampel
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    s = scorer.score(reference_summaries[idx], generated_summaries[idx])
    print(f"\n[ROUGE per sampel]: R1={s['rouge1'].fmeasure*100:.1f} | R2={s['rouge2'].fmeasure*100:.1f} | RL={s['rougeL'].fmeasure*100:.1f}")
    print("-" * 60)

In [ ]:
# Checklist kualitatif manual
# Isi sesuai hasil observasi dari sel di atas

qualitative_checklist = {
    "Mencakup tujuan penelitian": "?",  # Ganti: Ya / Sebagian / Tidak
    "Mencakup metode": "?",
    "Mencakup hasil utama": "?",
    "Mencakup kesimpulan": "?",
    "Koheren dan mudah dipahami": "?",
    "Ada indikasi hallucination": "?",   # Ganti: Ya / Tidak
    "Kehilangan konteks penting": "?",
}

print("\n📋 Checklist Evaluasi Kualitatif BART:")
print("-" * 40)
for k, v in qualitative_checklist.items():
    print(f"  {k:45s}: {v}")
print("\n⚠️  Isi checklist di atas setelah membaca hasil sampel di atas.")

---
## 7. Groundwork — Comparative Analysis

Sesuai proposal tahap 5: hasil BART disimpan untuk dibandingkan dengan model lain (TextRank, PEGASUS, LED, LongT5).

In [ ]:
import json

# Simpan semua prediksi dan referensi untuk comparative analysis
output_data = {
    "model": "PEGASUS",
    "model_name": MODEL_NAME,
    "dataset": "PubMed",
    "rouge_scores": rouge_scores,
    "num_test_samples": len(test_data),
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "predictions": generated_summaries,
    "references": reference_summaries
}

with open("pegasus_predictions.json", "w") as f:
    json.dump(output_data, f, indent=2)

print("Prediksi dan hasil disimpan ke: pegasus_predictions.json")
print("\n✅ Pipeline BART selesai! Summary:")
print(f"  Model    : {MODEL_NAME}")
print(f"  Dataset  : PubMed (ccdv/pubmed-summarization)")
print(f"  ROUGE-1  : {rouge_scores['ROUGE-1']}")
print(f"  ROUGE-2  : {rouge_scores['ROUGE-2']}")
print(f"  ROUGE-L  : {rouge_scores['ROUGE-L']}")
print("\n→ Lanjut ke notebook model berikutnya (TextRank / LongT5 / LED)!")

---
## 💾 Simpan Hasil ke Google Drive

In [ ]:
from google.colab import drive
import shutil, os, glob

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/My Drive/Binus/Semester_4/NLP/FinalProject/Result/PEGASUS'
os.makedirs(DRIVE_PATH, exist_ok=True)

# Simpan visualisasi
viz_files = glob.glob('./viz_*.png')
for f in viz_files:
    shutil.copy(f, DRIVE_PATH)
    print(f'  ✅ {f} → Drive')

# Simpan hasil evaluasi
shutil.copy('pegasus_rouge_results.csv', DRIVE_PATH)
shutil.copy('pegasus_predictions.json', DRIVE_PATH)
print(f'  ✅ pegasus_rouge_results.csv → Drive')
print(f'  ✅ pegasus_predictions.json → Drive')

# Simpan model weights (opsional)
DRIVE_MODEL_PATH = '/content/drive/My Drive/Binus/Semester_4/NLP/FinalProject/Models/PEGASUS'
os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)
if not os.path.exists(f'{DRIVE_MODEL_PATH}/pegasus-pubmed-finetuned'):
    print("Menyimpan model weights ke Drive... (~beberapa menit)")
    shutil.copytree('./pegasus-pubmed-finetuned', f'{DRIVE_MODEL_PATH}/pegasus-pubmed-finetuned')
    print(f'  ✅ Model weights → Drive')
else:
    print("Model weights sudah ada di Drive — skip")

print(f'\n✅ Semua hasil disimpan ke: {DRIVE_PATH}')
